In [ ]:
import requests
import time
import json
from datetime import datetime

API_URL = "https://api.example.com/v1/orders"
API_KEY = "1111"
REQUIRED_FIELDS = ["order_id", "customer_id", "order_date", "total_amount"]
MAX_RETRIES = 3
BACKOFF_SECONDS = 2

In [ ]:
def get_auth_headers():
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json",
    }

In [ ]:
def fetch_page(session, cursor):
    params = {"limit": 100}
    if cursor:
        params["cursor"] = cursor

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(API_URL, params=params, timeout=10)

            if response.status_code == 401:
                raise Exception("Auth failed") 

            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", BACKOFF_SECONDS * attempt))
                print(f"Rate limited, waiting {wait} seconds, attempt {attempt}")
                time.sleep(wait)
                continue

            response.raise_for_status()
            return response.json()

        except Exception as e:
            print(f"Other exception occurred: {e}")

In [ ]:
def extract(session):
    records = []
    cursor = None

    while True:
        page = fetch_page(session, cursor)
        batch = page.get("results", [])
        records.extend(batch)
        cursor = page.get("next_cursor")
        if not cursor:
            break

    return records

In [ ]:
def validate(record):
    for field in REQUIRED_FIELDS:
        if record.get(field) in (None, "", []):
            return False, f"Missing required field '{field}'"

    return True, None

In [ ]:
def run_pipeline():
    with requests.Session() as session:
        session.headers.update(get_auth_headers())

        raw_records = extract(session)